# 05 — Generalization: the gap between train and test

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/005-generalization.ipynb)

A model that scores perfectly on the data it was fitted to may still be useless on new data. This notebook fits the same twelve noisy sine points with polynomials of growing degree, measures the error on rows it saw and rows it did not, and shows the moment the two numbers part company.

## 1. The numbers: twelve points, split in two

The target is a sine wave with a little noise on top. Four of the twelve rows are held out from fitting — the model never sees their targets — so the error on them measures something the training error cannot: behaviour on data that does not exist yet.

In [ ]:
import numpy as np                                        # the array library
rng = np.random.default_rng(21)                           # fixed seed: the same noise on every run
x = np.linspace(0, 1, 12)                                 # twelve inputs spread across the range
y = np.sin(2 * np.pi * x) + rng.normal(0, 0.12, 12)       # one full sine wave plus measurement noise
idx = np.arange(12)                                       # row numbers 0 to 11
train = idx % 3 != 0                                      # eight rows used for fitting
test = ~train                                             # four rows kept back for grading
print("x:", np.round(x, 2))                               # the inputs
print("y:", np.round(y, 2))                               # the noisy targets
print("train x:", np.round(x[train], 2))                  # where the model may look
print("test  x:", np.round(x[test], 2))                   # where it is graded
print("counts: train", train.sum(), " test", test.sum())   # eight and four

**Line by line**

- `import numpy as np` — loads NumPy.
- `rng = np.random.default_rng(21)` — a seeded random generator, so results are reproducible.
- `x = np.linspace(0, 1, 12)` — twelve inputs from 0 to 1.
- `y = ...` — the sine of `2*pi*x`, plus Gaussian noise with spread 0.12.
- `idx = np.arange(12)` — row labels, used only to build the split.
- `train = idx % 3 != 0` — a boolean mask: True for the eight fitting rows.
- `test = ~train` — the complement: the four held-out rows.
- `print("x:" ...)` — the twelve inputs.
- `print("y:" ...)` — the noisy targets.
- `print("train x" ...)` — where the model may look.
- `print("test x" ...)` — confirms the split is interleaved, not one block.
- `print("counts" ...)` — eight rows for fitting, four for grading.

## 2. Step by step: three models, two errors, one gap

`np.polyfit` finds the polynomial of a chosen degree with the smallest squared error on the training rows. Raise the degree and the model gains freedom: a degree-9 curve has ten coefficients for eight points, so it can pass almost exactly through every one of them.

In [ ]:
import warnings                                           # keep NumPy's conditioning warning out of the way
warnings.filterwarnings("ignore")                         # degree 9 on 8 points is barely determined

for deg in [1, 3, 9]:                                     # a line, then a cubic, then a wiggle machine
    coef = np.polyfit(x[train], y[train], deg)             # least squares on the training rows only
    pred_train = np.polyval(coef, x[train])                # predictions on rows it learned from
    pred_test = np.polyval(coef, x[test])                  # predictions on rows it never saw
    err_train = np.mean((pred_train - y[train]) ** 2)       # mean squared error, training
    err_test = np.mean((pred_test - y[test]) ** 2)          # mean squared error, held out
    print(f"degree {deg}: train {err_train:.4f}  test {err_test:.4f}  gap {err_test - err_train:+.4f}")   # the gap is the story

**Line by line**

- `import warnings` — NumPy warns when a fit is nearly undetermined.
- `warnings.filterwarnings("ignore")` — silence that warning so the numbers stay readable.
- `for deg in [1, 3, 9]` — three complexities, from a straight line to a nine-degree curve.
- `coef = np.polyfit(...)` — fits the polynomial by least squares on the training rows.
- `pred_train = np.polyval(...)` — evaluates that polynomial at the training inputs.
- `pred_test = np.polyval(...)` — evaluates it at the held-out inputs.
- `err_train = ...` — average squared miss on rows the fit has seen.
- `err_test = ...` — average squared miss on rows it has not.
- `print(f"degree {deg} ...")` — both errors and their difference, side by side.

## 3. Plot: error against complexity

Sweep the degree from 1 to 9 and plot both errors on a log scale. The training curve only ever falls — more freedom always fits the sample better. The test curve is U-shaped, and its minimum is the complexity worth choosing.

In [ ]:
import matplotlib.pyplot as plt                          # plotting library
degrees = np.arange(1, 10)                               # every degree from 1 to 9
train_err = []                                           # one training error per degree
test_err = []                                            # one test error per degree
for d in degrees:                                        # one iteration per model complexity
    coef = np.polyfit(x[train], y[train], d)              # refit at this degree
    train_err.append(np.mean((np.polyval(coef, x[train]) - y[train]) ** 2))   # seen rows
    test_err.append(np.mean((np.polyval(coef, x[test]) - y[test]) ** 2))      # held-out rows
plt.figure(figsize=(6, 3))                               # a wide, short figure
plt.plot(degrees, train_err, "o-", color="tab:blue", label="train")   # the falling curve
plt.plot(degrees, test_err, "o-", color="tab:red", label="test")      # the U-shaped curve
plt.yscale("log")                                        # the errors span several orders of magnitude
plt.xlabel("polynomial degree (model freedom)")          # x-axis label
plt.ylabel("mean squared error, log scale")              # y-axis label
plt.legend()                                             # name the two lines
plt.title("More freedom lowers train error and raises test error")   # the takeaway
plt.show()                                               # render

**Line by line**

- `import matplotlib.pyplot as plt` — the plotting library.
- `degrees = np.arange(1, 10)` — 1 through 9.
- `train_err = []` — one training error per degree.
- `test_err = []` — one test error per degree.
- `for d in degrees` — one model per degree.
- `coef = np.polyfit(...)` — the best polynomial of that degree on the training rows.
- `train_err.append(...)` — training error stored in order.
- `test_err.append(...)` — held-out error stored in order.
- `plt.figure(...)` — a wide, short figure.
- `plt.plot(degrees, train_err, "o-")` — markers plus a line, one point per degree.
- `plt.plot(degrees, test_err, "o-")` — the same for the held-out error.
- `plt.yscale("log")` — a log axis, because test error jumps by a factor of ten.
- `plt.xlabel(...)` — model freedom, from low to high.
- `plt.ylabel(...)` — error, on a log scale.
- `plt.legend()` — names the two curves.
- `plt.title(...)` — what is being traded against what.
- `plt.show()` — renders the figure.

## 4. Interactive: drag the degree

The slider refits the polynomial at each degree. Blue points were used for fitting, green points were not, and the grey line is the noise-free sine nobody gets to see. Watch the red curve stop following the wave and start following the noise.

In [ ]:
from ipywidgets import interact                            # sliders that rerun a function
fine = np.linspace(0, 1, 200)                             # a dense grid across the data range
truth = np.sin(2 * np.pi * fine)                          # the noise-free pattern the data came from

def show(deg=3):                                         # deg comes from the slider
    coef = np.polyfit(x[train], y[train], deg)            # fit at the chosen complexity
    tr = np.mean((np.polyval(coef, x[train]) - y[train]) ** 2)   # training error
    te = np.mean((np.polyval(coef, x[test]) - y[test]) ** 2)     # held-out error
    plt.figure(figsize=(6, 3))                           # a fresh figure for each drag
    plt.plot(fine, truth, color="grey", alpha=0.6, label="true sine")          # the hidden pattern
    plt.plot(fine, np.polyval(coef, fine), color="tab:red", label=f"degree {deg}")   # the fitted curve
    plt.scatter(x[train], y[train], color="tab:blue", label="train")            # rows used for fitting
    plt.scatter(x[test], y[test], color="tab:green", s=70, label="test")        # rows held back
    plt.ylim(-2.5, 3)                                    # fixed axes so frames do not jump
    plt.legend(loc="upper right", fontsize=8)            # name every element
    plt.title(f"degree {deg}: train {tr:.4f} | test {te:.4f}")   # live readout
    plt.show()                                           # draw the frame

interact(show, deg=(1, 9, 1))                            # attach the slider

**Line by line**

- `from ipywidgets import interact` — the widget helper.
- `fine = np.linspace(-0.1, 1.1, 200)` — a dense grid, slightly wider than the data.
- `truth = np.sin(...)` — the noiseless sine, for reference only.
- `def show(deg=3)` — redraws one frame per slider value.
- `coef = np.polyfit(...)` — refits the model at that degree.
- `tr = ...` — training error, for the readout.
- `te = ...` — held-out error, for the readout.
- `plt.figure(...)` — a fresh figure per drag.
- `plt.plot(fine, truth, ...)` — the pattern the data was drawn from.
- `plt.plot(fine, np.polyval(coef, fine), ...)` — the model, evaluated on the dense grid.
- `plt.scatter(...)` — the blue training rows.
- `plt.scatter(...)` — the green held-out rows, drawn larger.
- `plt.ylim(-2.5, 3)` — clamps the view so high degrees do not blow up the axes.
- `plt.legend(...)` — names all four elements in a corner.
- `plt.title(f"...")` — live numbers, so you can read the gap as you drag.
- `plt.show()` — draws the frame.
- `interact(show, deg=(1, 9, 1))` — slider from degree 1 to 9, step 1.

## 5. When it breaks: degree 9 memorises the noise

At degree 9 the ten coefficients can satisfy eight training points almost exactly, so the training error collapses to zero — the model has learned the noise, not the wave. The four held-out points then land far from the curve, and the test error is the only number that warns you.

In [ ]:
coef9 = np.polyfit(x[train], y[train], 9)                # ten coefficients, eight points: almost exact
print("train error:", round(np.mean((np.polyval(coef9, x[train]) - y[train]) ** 2), 6))   # essentially zero
print("test  error:", round(np.mean((np.polyval(coef9, x[test]) - y[test]) ** 2), 4))      # many times larger
for j in np.where(test)[0]:                              # inspect every held-out row
    print(f"x={x[j]:.2f}  truth={y[j]:+.3f}  predicted={np.polyval(coef9, x[j]):+.3f}")   # where it misses
plt.figure(figsize=(6, 3))                               # a wide, short figure
plt.plot(fine, np.polyval(coef9, fine), color="tab:red", label="degree 9 fit")   # the memorising curve
plt.scatter(x[train], y[train], color="tab:blue", label="train")                 # points it was fitted to
plt.scatter(x[test], y[test], color="tab:green", s=70, label="test")             # points it must generalise to
plt.ylim(-2.5, 3)                                        # clip the extremes so the shape stays visible
plt.legend(loc="upper right", fontsize=8)                # name the elements
plt.title("Zero training error, large test error: the gap is overfitting")   # the takeaway
plt.show()                                               # render

**Line by line**

- `coef9 = np.polyfit(...)` — one polynomial of degree 9, fitted on eight rows.
- `print("train error" ...)` — rounded to six places because it is nearly zero.
- `print("test error" ...)` — the same model on rows it never saw.
- `for j in np.where(test)[0]` — loops over the four held-out row numbers.
- `print(f"x={x[j]:.2f} ...")` — truth and prediction on each held-out row.
- `plt.figure(...)` — a wide, short figure.
- `plt.plot(fine, np.polyval(coef9, fine), ...)` — the fitted curve across the whole range.
- `plt.scatter(...)` — the blue training points it memorised.
- `plt.scatter(...)` — the larger green test points it must generalise to.
- `plt.ylim(-2.5, 3)` — keeps the interesting part in frame.
- `plt.legend(...)` — names the curve and both point sets.
- `plt.title(...)` — names the failure: good on train, bad on test.
- `plt.show()` — renders the figure.

## Your turn

Change the split so that `test` holds out the two end points (`idx == 0` or `idx == 11`) and rerun section 2 — the gap for degree 9 should get much larger. Then find the degree with the lowest test error in section 4, and say why you cannot use that number to choose a model in real life.